# Experiment 6: SAE Feature Tracing Analysis

For each Exp6 trial where concept injection caused a leak, we re-ran the sentence through the model + SAE to trace which features activate.

**Phase 1**: Injection SAE trace — clean vs injected features at the injection layer (last token).
**Phase 2**: Reflection SAE trace — features active when processing the "Why did you mention X?" multi-turn prompt (no injection).

Key questions:
1. What features distinguish leaked from non-leaked trials? (Phase 1)
2. What features predict awareness vs confabulation reflections? (Phase 2)
3. Are there metacognition-relevant features, or only concept-content features?

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy import stats

# Load trace results
def load_trace(path):
    with open(path) as f:
        return json.load(f)

data_4b = load_trace('results/exp6_reflection/exp6_sae_trace.json')
try:
    data_27b = load_trace('results_27b/exp6_reflection/exp6_sae_trace.json')
except FileNotFoundError:
    data_27b = None
    print('27B results not found')

trials_4b = data_4b['trials']
trials_27b = data_27b['trials'] if data_27b else []

leaked_4b = [t for t in trials_4b if t['leaked']]
baseline_4b = [t for t in trials_4b if not t['leaked']]
leaked_27b = [t for t in trials_27b if t['leaked']]
baseline_27b = [t for t in trials_27b if not t['leaked']]

print(f'4B: {len(leaked_4b)} leaked, {len(baseline_4b)} baseline')
if trials_27b:
    print(f'27B: {len(leaked_27b)} leaked, {len(baseline_27b)} baseline')

# Grade distribution
grades_4b = Counter(t.get('gpt5_grade') or 'unknown' for t in leaked_4b)
print(f'\n4B GPT-5 grades: {dict(grades_4b)}')
if leaked_27b:
    grades_27b = Counter(t.get('gpt5_grade') or 'unknown' for t in leaked_27b)
    print(f'27B GPT-5 grades: {dict(grades_27b)}')

## Phase 1: Top Features by Mean |Delta| (Leaked Trials)

In [ ]:
def aggregate_delta_features(trials, top_n=20):
    """Aggregate delta features across trials, return top features by mean |delta|."""
    feature_deltas = defaultdict(list)  # feature_id -> list of delta values
    
    for t in trials:
        if t['phase1'] is None:
            continue
        for feat_id, delta in t['phase1']['top_delta']:
            feature_deltas[feat_id].append(delta)
    
    # Compute mean |delta| for each feature
    feature_stats = []
    for feat_id, deltas in feature_deltas.items():
        feature_stats.append({
            'feature_id': feat_id,
            'mean_abs_delta': np.mean(np.abs(deltas)),
            'mean_delta': np.mean(deltas),
            'count': len(deltas),
            'frac_trials': len(deltas) / len(trials),
        })
    
    feature_stats.sort(key=lambda x: x['mean_abs_delta'], reverse=True)
    return feature_stats[:top_n]

top_delta_4b = aggregate_delta_features(leaked_4b, top_n=20)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
feat_labels = [f"F{f['feature_id']}" for f in top_delta_4b]
mean_deltas = [f['mean_delta'] for f in top_delta_4b]
fracs = [f['frac_trials'] for f in top_delta_4b]

colors = ['#e74c3c' if d > 0 else '#3498db' for d in mean_deltas]
bars = ax.barh(range(len(feat_labels)), [f['mean_abs_delta'] for f in top_delta_4b],
               color=colors, alpha=0.8)

ax.set_yticks(range(len(feat_labels)))
ax.set_yticklabels(feat_labels)
ax.set_xlabel('Mean |Delta| (injected - clean)')
ax.set_title(f'4B: Top 20 SAE Features by Injection Delta (n={len(leaked_4b)} leaked trials)')
ax.invert_yaxis()

# Add count annotations
for i, f in enumerate(top_delta_4b):
    ax.text(f['mean_abs_delta'] + 0.01, i,
            f"{f['frac_trials']:.0%} of trials",
            va='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_phase1_top_delta.png', dpi=150)
plt.show()

print('\nTop 20 features by mean |delta|:')
for f in top_delta_4b:
    print(f"  F{f['feature_id']:>6d}: mean_delta={f['mean_delta']:+.3f}, "
          f"|delta|={f['mean_abs_delta']:.3f}, "
          f"in {f['frac_trials']:.0%} of trials (n={f['count']})")

## Phase 1: Leaked vs Baseline Feature Comparison

In [ ]:
def build_feature_vectors(trials, phase='phase1', feat_type='top_delta'):
    """Build a dict: feature_id -> list of values (one per trial).
    
    For delta features, values are the signed deltas.
    For clean/injected, values are activations.
    """
    feat_vals = defaultdict(lambda: [])
    for t in trials:
        if t[phase] is None:
            continue
        data = t[phase].get(feat_type, [])
        seen = set()
        for feat_id, val in data:
            feat_vals[feat_id].append(val)
            seen.add(feat_id)
    return feat_vals

# Compare injection deltas: leaked vs baseline
leaked_deltas = build_feature_vectors(leaked_4b, 'phase1', 'top_delta')
baseline_deltas = build_feature_vectors(baseline_4b, 'phase1', 'top_delta')

# Find features present in both groups and run t-test
all_feats = set(leaked_deltas.keys()) | set(baseline_deltas.keys())
comparison = []

for fid in all_feats:
    lk = leaked_deltas.get(fid, [])
    bl = baseline_deltas.get(fid, [])
    if len(lk) < 3 or len(bl) < 3:
        continue
    
    mean_leaked = np.mean(np.abs(lk))
    mean_baseline = np.mean(np.abs(bl))
    t_stat, p_val = stats.ttest_ind(np.abs(lk), np.abs(bl), equal_var=False)
    
    comparison.append({
        'feature_id': fid,
        'mean_leaked': mean_leaked,
        'mean_baseline': mean_baseline,
        'diff': mean_leaked - mean_baseline,
        'p_value': p_val,
        'log10_p': -np.log10(max(p_val, 1e-20)),
        'n_leaked': len(lk),
        'n_baseline': len(bl),
    })

comparison.sort(key=lambda x: x['p_value'])

# Volcano plot
fig, ax = plt.subplots(figsize=(10, 7))
diffs = [c['diff'] for c in comparison]
log_ps = [c['log10_p'] for c in comparison]
sig = [c['p_value'] < 0.05 for c in comparison]

ax.scatter([d for d, s in zip(diffs, sig) if not s],
           [p for p, s in zip(log_ps, sig) if not s],
           alpha=0.4, color='gray', s=20, label='ns')
ax.scatter([d for d, s in zip(diffs, sig) if s],
           [p for p, s in zip(log_ps, sig) if s],
           alpha=0.7, color='#e74c3c', s=40, label='p<0.05')

# Label top features
for c in comparison[:10]:
    ax.annotate(f"F{c['feature_id']}", (c['diff'], c['log10_p']),
                fontsize=7, ha='center', va='bottom')

ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axvline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Difference in mean |delta| (leaked - baseline)')
ax.set_ylabel('-log10(p-value)')
ax.set_title(f'4B Phase 1: Leaked vs Baseline Feature Deltas (n={len(leaked_4b)} vs {len(baseline_4b)})')
ax.legend()
plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_leaked_vs_baseline.png', dpi=150)
plt.show()

print(f'Features tested: {len(comparison)}')
print(f'Significant (p<0.05): {sum(1 for c in comparison if c["p_value"] < 0.05)}')
print(f'\nTop 10 by significance:')
for c in comparison[:10]:
    print(f"  F{c['feature_id']:>6d}: leaked={c['mean_leaked']:.3f}, "
          f"baseline={c['mean_baseline']:.3f}, "
          f"diff={c['diff']:+.3f}, p={c['p_value']:.4f}")

## Phase 2: Top Features During Reflection (All Leaked Trials)

In [ ]:
def aggregate_features(trials, phase='phase2', feat_type='top_features', top_n=20):
    """Aggregate feature activations across trials."""
    feature_acts = defaultdict(list)
    n_valid = 0
    
    for t in trials:
        if t[phase] is None:
            continue
        n_valid += 1
        for feat_id, act in t[phase].get(feat_type, []):
            feature_acts[feat_id].append(act)
    
    feature_stats = []
    for feat_id, acts in feature_acts.items():
        feature_stats.append({
            'feature_id': feat_id,
            'mean_activation': np.mean(acts),
            'std_activation': np.std(acts),
            'count': len(acts),
            'frac_trials': len(acts) / max(n_valid, 1),
        })
    
    feature_stats.sort(key=lambda x: x['mean_activation'], reverse=True)
    return feature_stats[:top_n], n_valid

top_p2_4b, n_p2 = aggregate_features(leaked_4b, 'phase2', 'top_features', top_n=20)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
feat_labels = [f"F{f['feature_id']}" for f in top_p2_4b]
activations = [f['mean_activation'] for f in top_p2_4b]
fracs = [f['frac_trials'] for f in top_p2_4b]

bars = ax.barh(range(len(feat_labels)), activations, color='#2ecc71', alpha=0.8)
ax.set_yticks(range(len(feat_labels)))
ax.set_yticklabels(feat_labels)
ax.set_xlabel('Mean Activation')
ax.set_title(f'4B Phase 2: Top 20 SAE Features During Reflection (n={n_p2} trials)')
ax.invert_yaxis()

for i, f in enumerate(top_p2_4b):
    ax.text(f['mean_activation'] + 0.01, i,
            f"{f['frac_trials']:.0%}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_phase2_top_features.png', dpi=150)
plt.show()

print(f'\nPhase 2 trials with data: {n_p2}')
print('Top 20 reflection features:')
for f in top_p2_4b:
    print(f"  F{f['feature_id']:>6d}: mean_act={f['mean_activation']:.3f}, "
          f"in {f['frac_trials']:.0%} of trials")

## Phase 2: Awareness vs Confabulation Features (4B)

In [ ]:
# Split by grade
awareness_trials = [t for t in leaked_4b if t.get('gpt5_grade') == 'awareness']
confab_trials = [t for t in leaked_4b if t.get('gpt5_grade') == 'confabulation']
puzzle_trials = [t for t in leaked_4b if t.get('gpt5_grade') == 'puzzlement']

print(f'Awareness: {len(awareness_trials)}, Confabulation: {len(confab_trials)}, Puzzlement: {len(puzzle_trials)}')

# Build feature vectors for each group
def group_features(trials, phase='phase2', feat_type='top_features'):
    """Return dict: feature_id -> list of activation values."""
    feats = defaultdict(list)
    for t in trials:
        if t[phase] is None:
            continue
        for fid, val in t[phase].get(feat_type, []):
            feats[fid].append(val)
    return feats

aware_feats = group_features(awareness_trials)
confab_feats = group_features(confab_trials)

# Compare: t-test on activation values for each feature
all_feat_ids = set(aware_feats.keys()) | set(confab_feats.keys())
grade_comparison = []

for fid in all_feat_ids:
    aw = aware_feats.get(fid, [])
    cf = confab_feats.get(fid, [])
    if len(aw) < 2 or len(cf) < 2:
        continue
    
    mean_aw = np.mean(aw)
    mean_cf = np.mean(cf)
    t_stat, p_val = stats.ttest_ind(aw, cf, equal_var=False)
    
    # Cohen's d
    pooled_std = np.sqrt((np.var(aw) * (len(aw)-1) + np.var(cf) * (len(cf)-1)) / 
                         (len(aw) + len(cf) - 2))
    d = (mean_aw - mean_cf) / pooled_std if pooled_std > 1e-8 else 0.0
    
    grade_comparison.append({
        'feature_id': fid,
        'mean_awareness': mean_aw,
        'mean_confabulation': mean_cf,
        'diff': mean_aw - mean_cf,
        'cohens_d': d,
        'p_value': p_val,
        'n_aware': len(aw),
        'n_confab': len(cf),
    })

grade_comparison.sort(key=lambda x: x['p_value'])

# Plot top discriminating features
top_disc = grade_comparison[:15]

fig, ax = plt.subplots(figsize=(12, 7))
y_pos = range(len(top_disc))
width = 0.35

ax.barh([y - width/2 for y in y_pos],
        [f['mean_awareness'] for f in top_disc],
        width, label='Awareness', color='#3498db', alpha=0.8)
ax.barh([y + width/2 for y in y_pos],
        [f['mean_confabulation'] for f in top_disc],
        width, label='Confabulation', color='#e74c3c', alpha=0.8)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"F{f['feature_id']}" for f in top_disc])
ax.set_xlabel('Mean Activation')
ax.set_title(f'4B Phase 2: Awareness vs Confabulation (n={len(awareness_trials)} vs {len(confab_trials)})')
ax.invert_yaxis()
ax.legend()

for i, f in enumerate(top_disc):
    sig = '*' if f['p_value'] < 0.05 else ''
    ax.text(max(f['mean_awareness'], f['mean_confabulation']) + 0.01, i,
            f"d={f['cohens_d']:.2f} p={f['p_value']:.3f}{sig}",
            va='center', fontsize=7)

plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_awareness_vs_confab.png', dpi=150)
plt.show()

print(f'\nFeatures compared: {len(grade_comparison)}')
print(f'Significant (p<0.05): {sum(1 for c in grade_comparison if c["p_value"] < 0.05)}')
print(f'\nTop 15 discriminating features (awareness vs confabulation):')
for f in top_disc:
    sig = ' *' if f['p_value'] < 0.05 else ''
    print(f"  F{f['feature_id']:>6d}: aware={f['mean_awareness']:.3f}, "
          f"confab={f['mean_confabulation']:.3f}, "
          f"d={f['cohens_d']:+.2f}, p={f['p_value']:.4f}{sig}")

## Phase 2: Awareness vs Puzzlement Features (4B)

In [ ]:
puzzle_feats = group_features(puzzle_trials)

# Compare awareness vs puzzlement
ap_comparison = []
for fid in set(aware_feats.keys()) | set(puzzle_feats.keys()):
    aw = aware_feats.get(fid, [])
    pz = puzzle_feats.get(fid, [])
    if len(aw) < 2 or len(pz) < 2:
        continue
    
    mean_aw = np.mean(aw)
    mean_pz = np.mean(pz)
    t_stat, p_val = stats.ttest_ind(aw, pz, equal_var=False)
    
    pooled_std = np.sqrt((np.var(aw) * (len(aw)-1) + np.var(pz) * (len(pz)-1)) / 
                         (len(aw) + len(pz) - 2))
    d = (mean_aw - mean_pz) / pooled_std if pooled_std > 1e-8 else 0.0
    
    ap_comparison.append({
        'feature_id': fid,
        'mean_awareness': mean_aw,
        'mean_puzzlement': mean_pz,
        'cohens_d': d,
        'p_value': p_val,
    })

ap_comparison.sort(key=lambda x: x['p_value'])

print(f'Features compared: {len(ap_comparison)}')
print(f'Significant (p<0.05): {sum(1 for c in ap_comparison if c["p_value"] < 0.05)}')
print(f'\nTop 10 discriminating features (awareness vs puzzlement):')
for f in ap_comparison[:10]:
    sig = ' *' if f['p_value'] < 0.05 else ''
    print(f"  F{f['feature_id']:>6d}: aware={f['mean_awareness']:.3f}, "
          f"puzzle={f['mean_puzzlement']:.3f}, "
          f"d={f['cohens_d']:+.2f}, p={f['p_value']:.4f}{sig}")

## Cross-Phase Correlation: Phase 1 Injection Features → Phase 2 Reflection Features

In [ ]:
# For each leaked trial, find shared features between Phase 1 delta and Phase 2 activation
shared_counts = []
p1_only_counts = []
p2_only_counts = []

for t in leaked_4b:
    if t['phase1'] is None or t['phase2'] is None:
        continue
    
    p1_feats = set(fid for fid, _ in t['phase1']['top_delta'][:50])
    p2_feats = set(fid for fid, _ in t['phase2']['top_features'][:50])
    
    shared = p1_feats & p2_feats
    shared_counts.append(len(shared))
    p1_only_counts.append(len(p1_feats - p2_feats))
    p2_only_counts.append(len(p2_feats - p1_feats))

print(f'Cross-phase overlap (top 50 features each):')
print(f'  Mean shared: {np.mean(shared_counts):.1f} / 50 ({np.mean(shared_counts)/50:.1%})')
print(f'  Mean Phase1-only: {np.mean(p1_only_counts):.1f}')
print(f'  Mean Phase2-only: {np.mean(p2_only_counts):.1f}')

# Histogram of overlap counts
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(shared_counts, bins=range(0, max(shared_counts) + 2), 
        color='#9b59b6', alpha=0.8, edgecolor='white')
ax.axvline(np.mean(shared_counts), color='red', linestyle='--', 
           label=f'Mean={np.mean(shared_counts):.1f}')
ax.set_xlabel('Number of Shared Features (top 50 each phase)')
ax.set_ylabel('Number of Trials')
ax.set_title('4B: Cross-Phase Feature Overlap')
ax.legend()
plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_cross_phase_overlap.png', dpi=150)
plt.show()

# Do awareness trials have more/less overlap than confabulation?
for grade_name, grade_trials in [('awareness', awareness_trials), 
                                   ('confabulation', confab_trials),
                                   ('puzzlement', puzzle_trials)]:
    overlaps = []
    for t in grade_trials:
        if t['phase1'] is None or t['phase2'] is None:
            continue
        p1_feats = set(fid for fid, _ in t['phase1']['top_delta'][:50])
        p2_feats = set(fid for fid, _ in t['phase2']['top_features'][:50])
        overlaps.append(len(p1_feats & p2_feats))
    if overlaps:
        print(f'  {grade_name}: mean overlap = {np.mean(overlaps):.1f} (n={len(overlaps)})')

## Per-Concept Breakdown

In [ ]:
# Group leaked trials by concept
concept_trials = defaultdict(list)
for t in leaked_4b:
    concept_trials[t['concept']].append(t)

print('Per-concept Phase 1 active features (injected):')
concept_data = []
for concept in sorted(concept_trials.keys()):
    trials = concept_trials[concept]
    mean_active = np.mean([t['phase1']['n_active_injected'] for t in trials if t['phase1']])
    mean_clean = np.mean([t['phase1']['n_active_clean'] for t in trials if t['phase1']])
    
    # Get top feature for this concept
    concept_deltas = defaultdict(list)
    for t in trials:
        if t['phase1']:
            for fid, delta in t['phase1']['top_delta'][:20]:
                concept_deltas[fid].append(abs(delta))
    
    top_feat = max(concept_deltas.items(), key=lambda x: np.mean(x[1])) if concept_deltas else (None, [])
    
    concept_data.append({
        'concept': concept,
        'n_trials': len(trials),
        'mean_active_clean': mean_clean,
        'mean_active_inj': mean_active,
        'top_feature': top_feat[0],
        'top_feat_mean_delta': np.mean(top_feat[1]) if top_feat[1] else 0,
    })
    print(f"  {concept:>10s}: n={len(trials):>3d}, "
          f"active_clean={mean_clean:.0f}, active_inj={mean_active:.0f}, "
          f"top_feat=F{top_feat[0]} (|d|={np.mean(top_feat[1]):.2f})" if top_feat[0] else '')

# Heatmap: concept x top features
# Get global top 10 features
global_top = aggregate_delta_features(leaked_4b, top_n=10)
global_top_ids = [f['feature_id'] for f in global_top]

concepts_sorted = sorted(concept_trials.keys())
heatmap_data = np.zeros((len(concepts_sorted), len(global_top_ids)))

for i, concept in enumerate(concepts_sorted):
    concept_deltas = defaultdict(list)
    for t in concept_trials[concept]:
        if t['phase1']:
            for fid, delta in t['phase1']['top_delta']:
                concept_deltas[fid].append(abs(delta))
    for j, fid in enumerate(global_top_ids):
        if fid in concept_deltas:
            heatmap_data[i, j] = np.mean(concept_deltas[fid])

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(global_top_ids)))
ax.set_xticklabels([f'F{fid}' for fid in global_top_ids], rotation=45, ha='right')
ax.set_yticks(range(len(concepts_sorted)))
ax.set_yticklabels(concepts_sorted)
ax.set_title('4B: Mean |Delta| by Concept x Feature (Top 10 Features)')
plt.colorbar(im, label='Mean |Delta|')
plt.tight_layout()
plt.savefig('results/exp6_reflection/exp6_sae_concept_heatmap.png', dpi=150)
plt.show()

## Summary Table

In [ ]:
print('=' * 80)
print('EXPERIMENT 6 SAE TRACE: SUMMARY')
print('=' * 80)

# Phase 1 summary
print(f'\n--- Phase 1: Injection Features ---')
print(f'Leaked trials: {len(leaked_4b)}')
print(f'Baseline trials: {len(baseline_4b)}')
n_sig_p1 = sum(1 for c in comparison if c['p_value'] < 0.05) if comparison else 0
print(f'Features significantly different (leaked vs baseline): {n_sig_p1}')
if top_delta_4b:
    print(f'Top feature by |delta|: F{top_delta_4b[0]["feature_id"]} '
          f'(mean |delta|={top_delta_4b[0]["mean_abs_delta"]:.3f})')

# Phase 2 summary
print(f'\n--- Phase 2: Reflection Features ---')
print(f'Trials with Phase 2: {n_p2}')
print(f'Awareness: {len(awareness_trials)}, Confabulation: {len(confab_trials)}, Puzzlement: {len(puzzle_trials)}')

n_sig_grade = sum(1 for c in grade_comparison if c['p_value'] < 0.05) if grade_comparison else 0
print(f'Features significantly different (awareness vs confab): {n_sig_grade}')

if grade_comparison:
    top_gc = grade_comparison[0]
    print(f'Most discriminating feature: F{top_gc["feature_id"]} '
          f'(d={top_gc["cohens_d"]:+.2f}, p={top_gc["p_value"]:.4f})')

# Cross-phase
print(f'\n--- Cross-Phase ---')
if shared_counts:
    print(f'Mean feature overlap (top 50 each): {np.mean(shared_counts):.1f} ({np.mean(shared_counts)/50:.0%})')

# Key question
print(f'\n--- Key Finding ---')
if n_sig_grade == 0:
    print('No SAE features significantly distinguish awareness from confabulation.')
    print('The model\'s reflection quality (awareness vs confabulation) does not appear')
    print('to be driven by distinct SAE-identifiable features at the injection layer.')
else:
    print(f'{n_sig_grade} features distinguish awareness from confabulation.')
    print('These may represent metacognition-relevant features worth further investigation.')

print('\n' + '=' * 80)